In [6]:
# Imports
import os
import re
import hashlib
import warnings
from urllib.parse import urlparse

import numpy as np
import pandas as pd

import email
from email import policy
from email.parser import Parser as EmailParser
from email.utils import parseaddr, getaddresses

warnings.filterwarnings("ignore")

# Sanity check that everything imported correctly
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("email module loaded OK:", email.__name__)
print("All imports successful.")

# Config — file paths

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/phishing project datasets"

PATH_NAZARIO   = os.path.join(BASE_DIR, "Nazario_5.csv")
PATH_EMAILTEXT = os.path.join(BASE_DIR, "email_text.csv")
PATH_PHISHTANK = os.path.join(BASE_DIR, "PhishTank_2026.csv")

OUT_DIR = os.path.join(BASE_DIR, "processed")
os.makedirs(OUT_DIR, exist_ok=True)

for p in [PATH_NAZARIO, PATH_EMAILTEXT, PATH_PHISHTANK]:
    print(p, "->", "FOUND" if os.path.exists(p) else "MISSING")

# Robust CSV loading + inspection

def load_csv_robust(path):
    """
    Try common encodings since older email dumps (Nazario/TREC07) are frequently
    NOT clean UTF-8. Falls back gracefully.
    """
    encodings_to_try = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]
    last_err = None
    for enc in encodings_to_try:
        try:
            df = pd.read_csv(path, encoding=enc, engine="python", on_bad_lines="warn")
            print(f"Loaded '{os.path.basename(path)}' with encoding={enc}")
            return df
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"Could not load {path} with any tried encoding. Last error: {last_err}")


def inspect_dataframe(df, name):
    print(f"\n{'='*70}\nINSPECTING: {name}\n{'='*70}")
    print("Shape:", df.shape)
    print("\nColumns:", list(df.columns))
    print("\nDtypes:\n", df.dtypes)
    print("\nNull counts:\n", df.isnull().sum())
    print("\nSample rows:")
    with pd.option_context("display.max_colwidth", 120):
        display(df.head(3))
    return df


df_nazario   = load_csv_robust(PATH_NAZARIO)
df_emailtext = load_csv_robust(PATH_EMAILTEXT)
df_phishtank = load_csv_robust(PATH_PHISHTANK)

inspect_dataframe(df_nazario, "Nazario_5.csv")
inspect_dataframe(df_emailtext, "email_text.csv")
inspect_dataframe(df_phishtank, "PhishTank_2026.csv")

# Column auto-detection

def detect_text_column(df, candidates=None):
    """
    Try to find the column holding raw email text/body. Returns the column name
    or None if nothing confident was found (caller should then inspect manually).
    """
    if candidates is None:
        candidates = ["sender", "receiver", "date", "subject", "body",
                      "label", "urls", "text"]
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    # fallback: pick the object/string column with the largest average string length
    obj_cols = df.select_dtypes(include="object").columns
    if len(obj_cols) == 0:
        return None
    avg_lens = {c: df[c].astype(str).str.len().mean() for c in obj_cols}
    best_col = max(avg_lens, key=avg_lens.get)
    print(f"No exact column-name match found. Best guess by avg text length: '{best_col}' "
          f"(avg len={avg_lens[best_col]:.1f}). Verify this is correct before proceeding.")
    return best_col


def detect_url_column(df, candidates=None):
    if candidates is None:
        candidates = ["url", "phish_url", "link", "website"]
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand in lower_map:
            return lower_map[cand]
    return None


NAZARIO_TEXT_COL   = detect_text_column(df_nazario)
EMAILTEXT_TEXT_COL = detect_text_column(df_emailtext)
PHISHTANK_URL_COL  = detect_url_column(df_phishtank)

print("\nDetected columns:")
print("  Nazario_5.csv text column   ->", NAZARIO_TEXT_COL)
print("  email_text.csv text column  ->", EMAILTEXT_TEXT_COL)
print("  PhishTank_2026.csv url col  ->", PHISHTANK_URL_COL)

# Deduplication

def normalize_for_hash(text):
    """Lowercase + collapse whitespace so near-identical dupes (differing only in
    whitespace/casing) are still caught."""
    if pd.isna(text):
        return ""
    return re.sub(r"\s+", " ", str(text).strip().lower())


def deduplicate_dataframe(df, subset_cols=None, text_col=None, verbose=True):
    """
    Removes exact duplicate rows first (fast path), then — if text_col is given —
    removes duplicates based on a normalized content hash (catches dupes that
    differ only in incidental whitespace/case).
    """
    before = len(df)
    df = df.copy()

    if subset_cols:
        df = df.drop_duplicates(subset=subset_cols)
    else:
        df = df.drop_duplicates()

    after_exact = len(df)

    if text_col and text_col in df.columns:
        df["_content_hash"] = df[text_col].apply(
            lambda x: hashlib.md5(normalize_for_hash(x).encode("utf-8")).hexdigest()
        )
        df = df.drop_duplicates(subset="_content_hash")
        df = df.drop(columns=["_content_hash"])

    after_all = len(df)

    if verbose:
        print(f"Deduplication: {before} -> {after_exact} (exact dupes removed) "
              f"-> {after_all} (content-hash dupes removed)")

    return df.reset_index(drop=True)

# Handling malformed / missing entries

def is_missing_or_malformed(text, min_length=5):
    """
    Flags a value as unusable if it's NaN, empty, whitespace-only, too short to be
    a meaningful email/body, or not a string at all.
    """
    if pd.isna(text):
        return True
    if not isinstance(text, str):
        text = str(text)
    stripped = text.strip()
    if len(stripped) < min_length:
        return True
    # Catch common placeholder junk values seen in scraped datasets
    if stripped.lower() in {"nan", "none", "null", "n/a", "na", "-"}:
        return True
    return False


def clean_dataframe(df, target_col, min_length=5, verbose=True):
    before = len(df)
    mask_bad = df[target_col].apply(lambda x: is_missing_or_malformed(x, min_length))
    n_bad = mask_bad.sum()
    df_clean = df.loc[~mask_bad].reset_index(drop=True)
    if verbose:
        print(f"Malformed/missing removal on '{target_col}': "
              f"{before} -> {len(df_clean)} ({n_bad} rows dropped)")
    return df_clean

# 7: Email header & metadata extraction

def parse_email_headers(raw_text):
    """
    Parses a raw email string using Python's email library and pulls out the
    standard headers + a few useful derived metadata fields.

    Handles two cases:
      1. raw_text actually contains RFC822-style headers (From:, Subject:, etc.)
      2. raw_text is just a plain body with no headers (common in pre-stripped
         datasets) — in that case headers_found=False and body=original text.
    """
    result = {
        "headers_found": False,
        "from_addr": None,
        "to_addr": None,
        "cc_addr": None,
        "subject": None,
        "date": None,
        "message_id": None,
        "return_path": None,
        "reply_to": None,
        "x_mailer": None,
        "content_type": None,
        "num_received_hops": 0,
        "num_attachments": 0,
        "body": raw_text,
    }

    if not isinstance(raw_text, str) or not raw_text.strip():
        return result

    # Quick heuristic: does this look like it has headers at all?
    looks_like_headers = bool(re.search(r"^\s*(From|To|Subject|Date|Received):",
                                         raw_text, re.MULTILINE | re.IGNORECASE))
    if not looks_like_headers:
        return result

    try:
        msg = email.message_from_string(raw_text, policy=policy.default)

        result["headers_found"] = True
        result["from_addr"] = msg.get("From")
        result["to_addr"] = msg.get("To")
        result["cc_addr"] = msg.get("Cc")
        result["subject"] = msg.get("Subject")
        result["date"] = msg.get("Date")
        result["message_id"] = msg.get("Message-ID")
        result["return_path"] = msg.get("Return-Path")
        result["reply_to"] = msg.get("Reply-To")
        result["x_mailer"] = msg.get("X-Mailer") or msg.get("User-Agent")
        result["content_type"] = msg.get_content_type()

        received_hops = msg.get_all("Received")
        result["num_received_hops"] = len(received_hops) if received_hops else 0

        # Attachment count (safe even for non-multipart messages)
        try:
            result["num_attachments"] = sum(1 for _ in msg.iter_attachments())
        except Exception:
            result["num_attachments"] = 0

        # Extract body text (plain preferred, falls back to html-stripped)
        try:
            body_part = msg.get_body(preferencelist=("plain", "html"))
            if body_part is not None:
                result["body"] = body_part.get_content()
        except Exception:
            pass

    except Exception as e:
        # Parsing failed outright — keep headers_found=False, body stays as raw_text
        result["parse_error"] = str(e)

    return result


def extract_headers_from_df(df, text_col):
    """Applies parse_email_headers row-wise and expands the results into new columns."""
    parsed = df[text_col].apply(parse_email_headers)
    parsed_df = pd.json_normalize(parsed)
    combined = pd.concat([df.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)
    n_with_headers = combined["headers_found"].sum()
    print(f"Header extraction on '{text_col}': {n_with_headers}/{len(combined)} rows had "
          f"parseable headers; the rest were treated as body-only text.")
    return combined

# 8: URL metadata extraction (for PhishTank)

IP_PATTERN = re.compile(r"^\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}$")

def extract_url_features(url):
    result = {
        "url_valid": False,
        "scheme": None,
        "domain": None,
        "is_ip_based": False,
        "subdomain_count": 0,
        "path_length": 0,
        "query_param_count": 0,
        "url_length": 0,
        "uses_https": False,
    }

    if not isinstance(url, str) or not url.strip():
        return result

    url = url.strip()
    result["url_length"] = len(url)

    try:
        parsed = urlparse(url if "://" in url else "http://" + url)
        result["url_valid"] = bool(parsed.netloc)
        result["scheme"] = parsed.scheme
        result["uses_https"] = parsed.scheme == "https"
        domain = parsed.netloc.split("@")[-1].split(":")[0]  # strip creds/port
        result["domain"] = domain
        result["is_ip_based"] = bool(IP_PATTERN.match(domain))
        result["subdomain_count"] = max(domain.count(".") - 1, 0) if not result["is_ip_based"] else 0
        result["path_length"] = len(parsed.path)
        result["query_param_count"] = len(parsed.query.split("&")) if parsed.query else 0
    except Exception:
        pass

    return result


def extract_url_features_df(df, url_col):
    parsed = df[url_col].apply(extract_url_features)
    parsed_df = pd.json_normalize(parsed)
    combined = pd.concat([df.reset_index(drop=True), parsed_df.reset_index(drop=True)], axis=1)
    n_valid = combined["url_valid"].sum()
    print(f"URL feature extraction on '{url_col}': {n_valid}/{len(combined)} rows had valid URLs.")
    return combined

# 9: Run the full pipeline on each dataset

# Nazario_5.csv
print("\n" + "#"*70 + "\n# Processing Nazario_5.csv\n" + "#"*70)
nz = df_nazario.copy()
nz = deduplicate_dataframe(nz, text_col=NAZARIO_TEXT_COL)
nz = clean_dataframe(nz, NAZARIO_TEXT_COL)
nz = extract_headers_from_df(nz, NAZARIO_TEXT_COL)
nz_out = os.path.join(OUT_DIR, "Nazario_5_processed.csv")
nz.to_csv(nz_out, index=False)
print("Saved ->", nz_out)

# email_text.csv
print("\n" + "#"*70 + "\n# Processing email_text.csv\n" + "#"*70)
et = df_emailtext.copy()
et = deduplicate_dataframe(et, text_col=EMAILTEXT_TEXT_COL)
et = clean_dataframe(et, EMAILTEXT_TEXT_COL)
et = extract_headers_from_df(et, EMAILTEXT_TEXT_COL)
et_out = os.path.join(OUT_DIR, "email_text_processed.csv")
et.to_csv(et_out, index=False)
print("Saved ->", et_out)

# PhishTank_2026.csv
print("\n" + "#"*70 + "\n# Processing PhishTank_2026.csv\n" + "#"*70)
pt = df_phishtank.copy()
if PHISHTANK_URL_COL is None:
    print("WARNING: could not auto-detect a URL column. Columns available:",
          list(pt.columns))
    print("Set PHISHTANK_URL_COL manually and re-run this cell.")
else:
    pt = deduplicate_dataframe(pt, text_col=PHISHTANK_URL_COL)
    pt = clean_dataframe(pt, PHISHTANK_URL_COL, min_length=4)
    pt = extract_url_features_df(pt, PHISHTANK_URL_COL)
    pt_out = os.path.join(OUT_DIR, "PhishTank_2026_processed.csv")
    pt.to_csv(pt_out, index=False)
    print("Saved ->", pt_out)

# 10: Quick post-processing sanity check

print("\nFinal shapes:")
print("  Nazario_5 processed:  ", nz.shape)
print("  email_text processed: ", et.shape)
if PHISHTANK_URL_COL is not None:
    print("  PhishTank processed:  ", pt.shape)

# 11: TESTS => run this to verify everything imported and works correctly

def run_tests():
    print("Running test suite...\n")

    # 1. Import sanity
    assert pd.__version__ is not None
    assert email.__name__ == "email"
    print("[PASS] Core imports available (pandas, numpy, email).")

    # 2. Deduplication
    toy = pd.DataFrame({
        "text": ["Hello world", "hello   world", "Something else", "Hello world"]
    })
    deduped = deduplicate_dataframe(toy, text_col="text", verbose=False)
    assert len(deduped) == 2, f"Expected 2 unique rows, got {len(deduped)}"
    print("[PASS] deduplicate_dataframe collapses whitespace/case near-dupes.")

    # 3. Malformed/missing detection
    assert is_missing_or_malformed(np.nan) is True
    assert is_missing_or_malformed("") is True
    assert is_missing_or_malformed("   ") is True
    assert is_missing_or_malformed("N/A") is True
    assert is_missing_or_malformed("hi") is True   # below min_length=5
    assert is_missing_or_malformed("This is a real email body.") is False
    print("[PASS] is_missing_or_malformed correctly flags edge cases.")

    # 4. Email header parsing: well-formed email
    sample_email = (
        "From: attacker@evil-example.com\n"
        "To: victim@example.com\n"
        "Subject: Urgent: Verify your account\n"
        "Date: Mon, 13 Jul 2026 10:00:00 -0000\n"
        "Message-ID: <abc123@evil-example.com>\n"
        "Received: from mail.evil-example.com by relay1.example.com\n"
        "Received: from relay1.example.com by mx.example.com\n"
        "Content-Type: text/plain\n"
        "\n"
        "Please click here to verify your account immediately."
    )
    parsed = parse_email_headers(sample_email)
    assert parsed["headers_found"] is True
    assert "attacker@evil-example.com" in parsed["from_addr"]
    assert parsed["subject"] == "Urgent: Verify your account"
    assert parsed["num_received_hops"] == 2
    assert "verify your account" in parsed["body"].lower()
    print("[PASS] parse_email_headers correctly extracts headers from well-formed email.")

    # 5. Email header parsing: body-only text (no headers)
    plain_body = "Hey, just checking in about the meeting tomorrow, let me know!"
    parsed_plain = parse_email_headers(plain_body)
    assert parsed_plain["headers_found"] is False
    assert parsed_plain["body"] == plain_body
    print("[PASS] parse_email_headers gracefully handles header-less body text.")

    # 6. Email header parsing: empty / NaN input
    parsed_empty = parse_email_headers(np.nan)
    assert parsed_empty["headers_found"] is False
    print("[PASS] parse_email_headers handles NaN input without crashing.")

    # 7. URL feature extraction
    f_https = extract_url_features("https://secure-login.example.com/account/verify")
    assert f_https["url_valid"] is True
    assert f_https["uses_https"] is True
    assert f_https["is_ip_based"] is False

    f_ip = extract_url_features("http://192.168.1.5/login.php")
    assert f_ip["is_ip_based"] is True
    assert f_ip["uses_https"] is False

    f_bad = extract_url_features(np.nan)
    assert f_bad["url_valid"] is False
    print("[PASS] extract_url_features correctly parses HTTPS, IP-based, and null URLs.")

    print("\nAll tests passed.")


run_tests()

pandas: 2.2.2
numpy: 2.0.2
email module loaded OK: email
All imports successful.
/content/drive/MyDrive/Colab Notebooks/phishing project datasets/Nazario_5.csv -> FOUND
/content/drive/MyDrive/Colab Notebooks/phishing project datasets/email_text.csv -> FOUND
/content/drive/MyDrive/Colab Notebooks/phishing project datasets/PhishTank_2026.csv -> FOUND
Loaded 'Nazario_5.csv' with encoding=utf-8
Loaded 'email_text.csv' with encoding=utf-8
Loaded 'PhishTank_2026.csv' with encoding=utf-8

INSPECTING: Nazario_5.csv
Shape: (3063, 7)

Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'label', 'urls']

Dtypes:
 sender      object
receiver    object
date        object
subject     object
body        object
label        int64
urls        object
dtype: object

Null counts:
 sender        2
receiver    112
date          3
subject      50
body          0
label         0
urls          0
dtype: int64

Sample rows:


,sender,receiver,date,subject,body,label,urls
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>, ""Brown, MeCole"" <MeCole.Brown@ENRON.com>, ""Cash, Michelle"" <Michelle...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report","User ID: enrondlr\nPW: bnaweb22\n\n\n -----Original Message-----\nFrom: \t""BNA Highlights"" <bhighlig@bna.com...",0,"['http://web.bna.com', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a0a4j5k3h4_', 'http://pubs.bna.com/ip/BNA/dlr.nsf/id/a..."
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a new report. Currently, only you and Jon have access to it. You'll se...",0,[]
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,"Rika r these new?\n\n -----Original Message-----\nFrom: \tThomas, Paul D. \nSent:\tFriday, June 29, 2001 8:39 AM\nT...",0,"['http://eastpower.dev.corp.enron.com/summary/pjmsummary.asp', 'http://eastpower.dev.corp.enron.com/summary/nyisosum..."



INSPECTING: email_text.csv
Shape: (53664, 2)

Columns: ['label', 'text']

Dtypes:
 label     int64
text     object
dtype: object

Null counts:
 label    0
text     0
dtype: int64

Sample rows:


,label,text
0,1,do you feel the pressure to perform and not rising to the occasion try v ia gr a your anxiety will be a thing of the...
1,0,hi i've just updated from the gulus and i check on other mirrors it seems there is a little typo in debian readme fi...
2,1,mega authenticv i a g r a discount pricec i a l i s discount pricedo not miss it click here http www moujsjkhchum com



INSPECTING: PhishTank_2026.csv
Shape: (64280, 2)

Columns: ['URL', 'Label']

Dtypes:
 URL      object
Label     int64
dtype: object

Null counts:
 URL      0
Label    0
dtype: int64

Sample rows:


,URL,Label
0,https://enjoyinn.cz/it/,1
1,https://koleckovyfest.cz/it/,1
2,https://koleckovyfest.cz/it,1



Detected columns:
  Nazario_5.csv text column   -> sender
  email_text.csv text column  -> label
  PhishTank_2026.csv url col  -> URL

######################################################################
# Processing Nazario_5.csv
######################################################################
Deduplication: 3063 -> 3063 (exact dupes removed) -> 2212 (content-hash dupes removed)
Malformed/missing removal on 'sender': 2212 -> 2211 (1 rows dropped)
Header extraction on 'sender': 0/2211 rows had parseable headers; the rest were treated as body-only text.
Saved -> /content/drive/MyDrive/Colab Notebooks/phishing project datasets/processed/Nazario_5_processed.csv

######################################################################
# Processing email_text.csv
######################################################################
Deduplication: 53664 -> 53664 (exact dupes removed) -> 2 (content-hash dupes removed)
Malformed/missing removal on 'label': 2 -> 0 (2 rows dropped)


KeyError: 'headers_found'

In [8]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/phishing project datasets/processed/Nazario_5_processed.csv")
df.head()

,sender,receiver,date,subject,body,label,urls,headers_found,from_addr,to_addr,...,subject.1,date.1,message_id,return_path,reply_to,x_mailer,content_type,num_received_hops,num_attachments,body.1
0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>","""Acevedo, Felecia"" <Felecia.Acevedo@ENRON.com>...","Fri, 29 Jun 2001 08:36:09 -0500","FW: June 29 -- BNA, Inc. Daily Labor Report",User ID: enrondlr\nPW: bnaweb22\n\n\n ...,0,"['http://web.bna.com', 'http://pubs.bna.com/ip...",False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,"""Hu, Sylvia"" <Sylvia.Hu@ENRON.com>"
1,"""Webb, Jay"" <Jay.Webb@ENRON.com>","""Lambie, Chris"" <Chris.Lambie@ENRON.com>","Fri, 29 Jun 2001 09:37:04 -0500",NGX failover plan.,"\nHi Chris, \n\nTonight we are rolling out a ...",0,[],False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,"""Webb, Jay"" <Jay.Webb@ENRON.com>"
2,"""Symms, Mark"" <Mark.Symms@ENRON.com>","""Thomas, Paul D."" <Paul.D.Thomas@ENRON.com>","Fri, 29 Jun 2001 08:39:30 -0500",RE: Intranet Site,Rika r these new?\n\n -----Original Message---...,0,['http://eastpower.dev.corp.enron.com/summary/...,False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,"""Symms, Mark"" <Mark.Symms@ENRON.com>"
3,"""Thorne, Judy"" <Judy.Thorne@ENRON.com>","""Grass, John"" <John.Grass@ENRON.com>, ""Nemec, ...","Fri, 29 Jun 2001 10:35:17 -0500",FW: ENA Upstream Company information,"John/Gerald,\n\nWe are currently trading under...",0,[],False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,"""Thorne, Judy"" <Judy.Thorne@ENRON.com>"
4,"""Williams, Jason R (Credit)"" <Jason.R.Williams...","""Nemec, Gerald"" <Gerald.Nemec@ENRON.com>, ""Dic...","Fri, 29 Jun 2001 10:40:02 -0500",New Master Physical,Gerald and Stacy -\n\nAttached is a worksheet ...,0,[],False,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,"""Williams, Jason R (Credit)"" <Jason.R.Williams..."


In [9]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/phishing project datasets/processed/email_text_processed.csv")
df.head()

,label,text,headers_found,from_addr,to_addr,cc_addr,subject,date,message_id,return_path,reply_to,x_mailer,content_type,num_received_hops,num_attachments,body
0,1,do you feel the pressure to perform and not ri...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,do you feel the pressure to perform and not ri...
1,0,hi i've just updated from the gulus and i chec...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,hi i've just updated from the gulus and i chec...
2,1,mega authenticv i a g r a discount pricec i a ...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,mega authenticv i a g r a discount pricec i a ...
3,1,hey billy it was really fun going out the othe...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,hey billy it was really fun going out the othe...
4,1,system of the home it will have the capabiliti...,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,system of the home it will have the capabiliti...


In [11]:
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/phishing project datasets/processed/PhishTank_2026_processed.csv")
df.head()

,URL,Label,url_valid,scheme,domain,is_ip_based,subdomain_count,path_length,query_param_count,url_length,uses_https
0,https://enjoyinn.cz/it/,1,True,https,enjoyinn.cz,False,0,4,0,23,True
1,https://koleckovyfest.cz/it/,1,True,https,koleckovyfest.cz,False,0,4,0,28,True
2,https://koleckovyfest.cz/it,1,True,https,koleckovyfest.cz,False,0,3,0,27,True
3,https://ca-espacefrance.com/,1,True,https,ca-espacefrance.com,False,0,1,0,28,True
4,https://ca-espacefrance.com/start.php,1,True,https,ca-espacefrance.com,False,0,10,0,37,True
